# Phase 2 — Topological Feature Extraction (Persistent Homology)

Computes $H_0$, $H_1$, $H_2$ persistent homology on the local atomic
neighbourhood of every residue, vectorises each persistence diagram into a
fixed-length descriptor, and concatenates it onto `data.x`.

**Input:** `phase1_graphs.pt` (mounted from the Phase 1 notebook output)
**Output:** `phase2_graphs.pt` — same graphs, node features widened 27 → 62

---

### Why the biology motivates this

A binding pocket *is* a topological feature. It is a concavity — in persistent
homology terms, a 1-cycle that persists across a range of scales ($H_1$), or an
enclosed void ($H_2$). Plain geometric features (coordinates, distances) encode
this only implicitly; persistence measures it directly and at every scale at
once. That is the actual novelty claim of this project, so it needs to survive
scrutiny, which is what the QC cell at the end is for.

### Three decisions that matter

1. **Alpha complex, not Vietoris–Rips.** For point clouds in $\mathbb{R}^3$ the
   alpha complex is a subcomplex of the Delaunay triangulation: it gives
   *identical* homology to the Čech complex but stays near-linear in size. A Rips
   complex expanded to dimension 3 (needed for $H_2$) on a 150-atom neighbourhood
   blows up to millions of simplices and is not tractable for 10k residues. Rips
   is retained only as a degenerate-input fallback.

2. **All heavy atoms, not just $C_\alpha$.** Pockets are formed by *side-chain*
   packing. A $C_\alpha$-only cloud has one bead per residue and simply cannot
   resolve a cavity at the relevant scale. Phase 1 saved only $C_\alpha$
   coordinates, so this notebook re-parses the PDB files to recover full
   heavy-atom clouds — a deliberate cost, paid for materially better topology.

3. **GUDHI returns *squared* filtration values for alpha complexes.** Every
   value is square-rooted here so births, deaths and persistences are in
   ångströms. Skipping this silently changes the scale of every feature and makes
   the radii thresholds meaningless.

## Cell 1 — Installs

In [ ]:
# Kaggle: Internet ON (re-parsing needs the PDB files). Accelerator irrelevant --
# persistent homology is CPU-only; nothing in this notebook touches the GPU.

!pip install -q torch-geometric==2.6.1
!pip install -q gudhi==3.10.1
!pip install -q biopython==1.84

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import gudhi
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", module="Bio")

print(f"gudhi  : {gudhi.__version__}")
print(f"torch  : {torch.__version__}")
print(f"CPUs   : {os.cpu_count()}")

## Cell 2 — Load the Phase 1 graphs

Searches the Kaggle input mounts first (Phase 1 attached as a notebook-output
data source), then falls back to `/kaggle/working` for a same-session run.

`weights_only=False` is required: PyG `Data` objects are pickled classes, and
torch ≥ 2.6 defaults to `weights_only=True`, which refuses to unpickle them.

In [ ]:
def locate_phase1() -> Path:
    """Find phase1_graphs.pt across Kaggle input mounts or the working dir."""
    candidates: List[Path] = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if root.exists():
            candidates.extend(root.rglob("phase1_graphs.pt"))
    if not candidates:
        raise FileNotFoundError(
            "phase1_graphs.pt not found. Attach the Phase 1 notebook via "
            "'+ Add Data -> Notebook Output', or run Phase 1 in this session."
        )
    best = max(candidates, key=lambda p: p.stat().st_size)
    print(f"found {len(candidates)} candidate(s); using {best}")
    return best


blob = torch.load(locate_phase1(), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
NODE_FEAT_DIM_P1 = blob["node_feat_dim"]

print(f"chains           : {len(dataset)}")
print(f"residues         : {sum(d.num_nodes for d in dataset):,}")
print(f"node feat dim    : {NODE_FEAT_DIM_P1}")
print(f"pos_weight       : {blob['pos_weight']:.2f}")
print(f"positives        : {sum(int(d.y.sum()) for d in dataset):,}")

## Cell 3 — Recover heavy-atom clouds

Re-parses each structure to get every heavy atom, grouped per residue, in the
**exact same residue order** Phase 1 used. The assertion at the end is the whole
point of this cell: if the ordering drifts by even one residue, every
topological feature is attached to the wrong node and the error is completely
invisible downstream.

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

from Bio.PDB import MMCIFParser, PDBParser

RAW = Path("/tmp/pdb_raw")
RAW.mkdir(parents=True, exist_ok=True)

AA3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V", "MSE": "M",
}


def fetch(pid: str) -> Optional[Path]:
    for ext in ("pdb", "cif"):
        p = RAW / f"{pid}.{ext}"
        if p.exists() and p.stat().st_size > 1000:
            return p
    for ext in ("pdb", "cif"):
        try:
            r = requests.get(f"https://files.rcsb.org/download/{pid}.{ext}", timeout=30)
            if r.status_code == 200 and len(r.content) > 1000:
                p = RAW / f"{pid}.{ext}"
                p.write_bytes(r.content)
                return p
        except requests.RequestException:
            continue
    return None


needed = sorted({d.pdb_id for d in dataset})
with ThreadPoolExecutor(max_workers=8) as ex:
    paths = dict(zip(needed, ex.map(fetch, needed)))
missing = [k for k, v in paths.items() if v is None]
print(f"structures available: {len(needed) - len(missing)}/{len(needed)}")
if missing:
    print(f"  MISSING: {missing}")


def heavy_atoms_by_chain(pdb_id: str, path: Path):
    """{chain_id: (per-residue heavy-atom arrays, (N,3) CA coords)} in Phase 1 order."""
    parser = MMCIFParser(QUIET=True) if path.suffix == ".cif" else PDBParser(QUIET=True, PERMISSIVE=True)
    model = next(parser.get_structure(pdb_id, str(path)).get_models())

    out = {}
    for chain in model:
        # Predicate MUST match Phase 1 exactly: standard residue AND has a CA.
        residues = [r for r in chain
                    if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        if not residues:
            continue
        per_res, cas = [], []
        for res in residues:
            xyz = [a.coord for a in res.get_unpacked_list() if a.element not in ("H", "D")]
            per_res.append(np.asarray(xyz, dtype=np.float32).reshape(-1, 3))
            # Take CA by NAME. Atom order in a PDB residue is N, CA, C, O, ...
            # so per_res[i][0] is the backbone nitrogen, not the alpha carbon.
            cas.append(np.asarray(res["CA"].coord, dtype=np.float32))
        out[chain.id] = (per_res, np.stack(cas, axis=0))
    return out


atom_cache = {}
for pid in needed:
    if paths[pid] is not None:
        atom_cache[pid] = heavy_atoms_by_chain(pid, paths[pid])

# --- alignment check ---------------------------------------------------------
aligned, dropped = [], []
for d in dataset:
    entry = atom_cache.get(d.pdb_id, {}).get(d.chain_id)
    if entry is None or len(entry[0]) != d.num_nodes:
        dropped.append((d.pdb_id, d.chain_id,
                        None if entry is None else len(entry[0]), d.num_nodes))
        continue
    per_res, ca = entry
    aligned.append((d, per_res, ca))

print(f"\naligned chains   : {len(aligned)}/{len(dataset)}")
if dropped:
    print("  MISALIGNED (excluded):")
    for pid, cid, got, want in dropped:
        print(f"    {pid}_{cid}: got {got} residues, expected {want}")

assert len(aligned) > 0, "no chains aligned -- residue selection predicate drifted"

# Cross-check against Phase 1 geometry: CA-CA distance matrices must agree,
# which proves the residue ORDER matches and not merely the residue count.
d0, _, ca0 = aligned[0]
p1 = d0.pos.numpy()
p1 = p1 - p1.mean(0)
p2 = ca0 - ca0.mean(0)
drift = float(np.abs(np.linalg.norm(p1[:, None] - p1[None], axis=-1)
                     - np.linalg.norm(p2[:, None] - p2[None], axis=-1)).max())
print(f"CA geometry check: max pairwise-distance drift vs Phase 1 = {drift:.2e} A")
assert drift < 1e-2, "residue ORDER differs from Phase 1 -- features would misalign"

n_atoms_tot = sum(sum(len(a) for a in pr) for _, pr, _ in aligned)
print(f"heavy atoms      : {n_atoms_tot:,} "
      f"({n_atoms_tot / sum(d.num_nodes for d, _, _ in aligned):.1f} per residue)")

## Cell 4 — Persistence computation and vectorisation

For each residue: take every heavy atom within `NBHD_RADIUS` of its $C_\alpha$,
build an alpha complex, extract the $H_0/H_1/H_2$ diagrams, and reduce each to
11 scalars.

**Per homology dimension (11 features):** bar count, total / max / mean / std
persistence, persistent entropy, mean birth, mean death, and Betti numbers at
3 fixed radii. Times 3 dimensions = 33, plus 2 local-density features = **35**.

Persistent entropy $E = -\sum_i p_i \log p_i$ with $p_i = \ell_i / \sum_j \ell_j$
measures how *concentrated* the topological signal is: one dominant cavity gives
low entropy, diffuse noise gives high entropy. It is the feature most likely to
separate a real pocket from surface roughness.

Degenerate inputs (fewer than 5 atoms, or coplanar sets that make Delaunay fail)
fall back to a Rips complex capped at dimension 2. This is rare but must be
handled — one QhullError otherwise kills a multi-minute job.

In [ ]:
MAX_HOM_DIM = 2
BETTI_RADII = (4.0, 6.0, 8.0)      # angstrom, where Betti curves are sampled
NBHD_RADIUS = 12.0                 # angstrom around the residue's CA
MAX_NBHD_ATOMS = 300               # deterministic cap: nearest-N, for runtime
FEATS_PER_DIM = 6 + 2 + len(BETTI_RADII)          # 11
TOPO_DIM = FEATS_PER_DIM * (MAX_HOM_DIM + 1) + 2  # 35

TOPO_NAMES = [
    f"H{d}_{s}" for d in range(MAX_HOM_DIM + 1)
    for s in ["n_bars", "total_pers", "max_pers", "mean_pers", "std_pers",
              "entropy", "mean_birth", "mean_death",
              *[f"betti_r{r:g}" for r in BETTI_RADII]]
] + ["nbhd_n_atoms", "nbhd_gyration"]
assert len(TOPO_NAMES) == TOPO_DIM


def _diagrams(points: np.ndarray) -> Dict[int, np.ndarray]:
    """H0/H1/H2 persistence diagrams, filtration values in ANGSTROM."""
    pts = np.ascontiguousarray(points, dtype=np.float64)
    st = None
    if len(pts) >= 5:
        try:
            ac = gudhi.AlphaComplex(points=pts)
            st = ac.create_simplex_tree()
            squared = True                     # GUDHI alpha default
        except Exception:
            st = None
    if st is None:                             # degenerate / coplanar fallback
        rc = gudhi.RipsComplex(points=pts, max_edge_length=float(NBHD_RADIUS))
        st = rc.create_simplex_tree(max_dimension=MAX_HOM_DIM + 1)
        squared = False

    st.compute_persistence(homology_coeff_field=2, persistence_dim_max=True)

    out = {}
    for d in range(MAX_HOM_DIM + 1):
        bars = st.persistence_intervals_in_dimension(d)
        bars = np.asarray(bars, dtype=np.float64).reshape(-1, 2)
        if squared and len(bars):
            # GUDHI returns SQUARED circumradii for alpha complexes. Convert to
            # angstroms, or every threshold below is silently on the wrong scale.
            bars = np.sqrt(np.maximum(bars, 0.0))
        out[d] = bars
    return out


def _vectorize(dgms: Dict[int, np.ndarray], pts: np.ndarray) -> np.ndarray:
    feats = []
    for d in range(MAX_HOM_DIM + 1):
        bars = dgms.get(d, np.zeros((0, 2)))
        finite = bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1, 2)

        if len(finite) == 0:
            feats.extend([0.0] * FEATS_PER_DIM)
            continue

        births, deaths = finite[:, 0], finite[:, 1]
        pers = np.maximum(deaths - births, 0.0)
        total = float(pers.sum())

        if total > 0:
            p = pers / total
            nz = p[p > 0]
            entropy = float(-(nz * np.log(nz)).sum())
        else:
            entropy = 0.0

        betti = [float(((births <= r) & (deaths > r)).sum()) for r in BETTI_RADII]

        feats.extend([
            float(len(finite)), total, float(pers.max()), float(pers.mean()),
            float(pers.std()), entropy, float(births.mean()), float(deaths.mean()),
            *betti,
        ])

    centred = pts - pts.mean(axis=0, keepdims=True)
    gyration = float(np.sqrt((centred ** 2).sum(axis=1).mean())) if len(pts) else 0.0
    feats.extend([float(len(pts)), gyration])
    return np.asarray(feats, dtype=np.float32)


def topo_features_for_chain(ca_xyz: np.ndarray, per_res_atoms: List[np.ndarray]) -> np.ndarray:
    """(N, TOPO_DIM) topological descriptors, one row per residue."""
    all_atoms = np.concatenate([a for a in per_res_atoms if len(a)], axis=0) \
        if any(len(a) for a in per_res_atoms) else np.zeros((0, 3), np.float32)
    if len(all_atoms) == 0:
        return np.zeros((len(ca_xyz), TOPO_DIM), np.float32)

    tree = cKDTree(all_atoms)
    out = np.zeros((len(ca_xyz), TOPO_DIM), dtype=np.float32)

    for i, ca in enumerate(ca_xyz):
        idx = tree.query_ball_point(ca, NBHD_RADIUS)
        if len(idx) > MAX_NBHD_ATOMS:          # deterministic: keep the nearest
            d, idx = tree.query(ca, k=MAX_NBHD_ATOMS)
            idx = np.asarray(idx)
        pts = all_atoms[np.asarray(idx, dtype=int)]
        if len(pts) < 4:
            out[i] = 0.0
            continue
        out[i] = _vectorize(_diagrams(pts), pts)
    return out

## Cell 5 — Run it

Parallelised across cores with `joblib`. Chains are the unit of work, so each
worker builds one KD-tree and reuses it for every residue in that chain.

In [ ]:
from joblib import Parallel, delayed

t0 = time.time()

# CA coords and the atom cloud come from the same parse, so they share a frame.
# Topology depends only on pairwise distances, so the frame itself is irrelevant.
n_jobs = max(1, (os.cpu_count() or 2))
print(f"computing persistent homology on {sum(d.num_nodes for d, _, _ in aligned):,} "
      f"residues with {n_jobs} workers ...")

topo_all = Parallel(n_jobs=n_jobs, verbose=1)(
    delayed(topo_features_for_chain)(ca, pr) for _, pr, ca in aligned
)

elapsed = time.time() - t0
n_res = sum(len(t) for t in topo_all)
print(f"\ndone in {elapsed:.1f}s  ({1000 * elapsed / max(n_res, 1):.1f} ms/residue)")

## Cell 6 — Fuse, validate, save

The three checks here, in order of importance:

1. **Rotation invariance of the descriptors.** Persistent homology depends only
   on pairwise distances, so these features *must* be invariant under SO(3). If
   they are not, something is broken — and since Phase 3 fuses them into an
   equivariant network, a non-invariant scalar feature would break equivariance
   for the whole model.
2. **Does topology carry signal at all?** Single-feature ROC-AUC against the
   binding-site labels. A feature at AUC ≈ 0.5 is noise. If *every* topological
   feature sits at 0.5, the premise of the project is not supported and you need
   to know that now, not after Phase 5.
3. **Degenerate-column check.** Constant or all-zero columns waste parameters
   and can destabilise normalisation.

Features are standardised (z-scored) using statistics computed over all
residues, and the scaler is saved — Phase 5 must apply the *training-split*
statistics, so this is a starting point, not the final normalisation.

In [ ]:
from sklearn.metrics import roc_auc_score

# --- assemble ----------------------------------------------------------------
topo_cat = np.concatenate(topo_all, axis=0)
y_cat = np.concatenate([d.y.numpy() for d, _, _ in aligned], axis=0)
print(f"topo matrix      : {topo_cat.shape}")

mu = topo_cat.mean(axis=0)
sd = topo_cat.std(axis=0)
sd[sd < 1e-6] = 1.0                     # guard constant columns

# --- 1. rotation invariance --------------------------------------------------
_, probe_atoms, probe_ca = aligned[0]
g = torch.Generator().manual_seed(0)
Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g))
Q = (Q * torch.sign(torch.det(Q))).numpy().astype(np.float32)
shift = np.array([31.7, -12.4, 88.1], dtype=np.float32)

sub, ca_sub = probe_atoms[:40], probe_ca[:40]
f_ref = topo_features_for_chain(ca_sub, sub)
f_rot = topo_features_for_chain(ca_sub @ Q.T + shift, [a @ Q.T + shift for a in sub])

scale = np.maximum(np.abs(f_ref).max(axis=0), 1e-6)
rel = float((np.abs(f_ref - f_rot) / scale).max())
print(f"rotation invariance: max relative deviation {rel:.2e}")
assert rel < 1e-3, "topological features are NOT rotation invariant"

# --- 2. does topology carry signal? -----------------------------------------
aucs = []
for j in range(TOPO_DIM):
    col = topo_cat[:, j]
    aucs.append(0.5 if col.std() < 1e-9 else roc_auc_score(y_cat, col))
aucs = np.asarray(aucs)
order = np.argsort(-np.abs(aucs - 0.5))

print("\ntop single-feature ROC-AUC vs binding-site label:")
for j in order[:8]:
    print(f"   {TOPO_NAMES[j]:22s} AUC {aucs[j]:.3f}")
print(f"   ... best |AUC-0.5| = {np.abs(aucs - 0.5).max():.3f}  "
      f"({(np.abs(aucs - 0.5) > 0.05).sum()}/{TOPO_DIM} features above 0.55/0.45)")

# --- 3. degenerate columns ---------------------------------------------------
dead = [TOPO_NAMES[j] for j in range(TOPO_DIM) if topo_cat[:, j].std() < 1e-9]
print(f"\nconstant columns : {len(dead)}" + (f"  {dead}" if dead else ""))

# --- fuse onto x -------------------------------------------------------------
out_ds = []
for (d, _, _), topo in zip(aligned, topo_all):
    z = (topo - mu) / sd
    d.x_topo = torch.from_numpy(topo.astype(np.float32))          # raw, for ablation
    d.x = torch.cat([d.x, torch.from_numpy(z.astype(np.float32))], dim=1)
    assert torch.isfinite(d.x).all(), f"{d.pdb_id}_{d.chain_id}: non-finite features"
    out_ds.append(d)

NODE_FEAT_DIM = out_ds[0].x.size(1)
print(f"\nnode feature dim : {NODE_FEAT_DIM_P1} -> {NODE_FEAT_DIM}  (+{TOPO_DIM} topological)")

out_dir = Path("/kaggle/working/processed")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "phase2_graphs.pt"
torch.save({
    "dataset": out_ds,
    "node_feat_dim": NODE_FEAT_DIM,
    "edge_feat_dim": out_ds[0].edge_attr.size(1),
    "topo_dim": TOPO_DIM,
    "topo_names": TOPO_NAMES,
    "topo_mean": mu, "topo_std": sd,
    "topo_auc": aucs,
    "pos_weight": blob["pos_weight"],
    "aa_alphabet": blob["aa_alphabet"],
}, out_path)
print(f"saved -> {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

## Cell 7 — Inspect the most discriminative descriptor

In [ ]:
import matplotlib.pyplot as plt

best = int(np.argmax(np.abs(aucs - 0.5)))
col = topo_cat[:, best]
pos, neg = col[y_cat > 0], col[y_cat == 0]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
lo, hi = np.percentile(col, [0.5, 99.5])
bins = np.linspace(lo, hi, 50)
ax[0].hist(neg, bins=bins, alpha=0.55, density=True, label=f"non-site (n={len(neg)})")
ax[0].hist(pos, bins=bins, alpha=0.55, density=True, label=f"binding site (n={len(pos)})")
ax[0].set_title(f"{TOPO_NAMES[best]}  (AUC {aucs[best]:.3f})")
ax[0].set_xlabel("feature value"); ax[0].legend(fontsize=8)

k = min(15, TOPO_DIM)
top = order[:k][::-1]
ax[1].barh(range(k), aucs[top] - 0.5)
ax[1].set_yticks(range(k)); ax[1].set_yticklabels([TOPO_NAMES[j] for j in top], fontsize=7)
ax[1].axvline(0, color="k", lw=0.8)
ax[1].set_xlabel("AUC - 0.5"); ax[1].set_title("most discriminative descriptors")
plt.tight_layout(); plt.show()

---

## Reading the results

**Rotation invariance must pass.** It is an assert, so a failure stops the
notebook. This is non-negotiable: Phase 3 treats these as invariant scalars.

**Single-feature AUC is the honest test of the premise.** Individual
topological descriptors are weak learners — an AUC of 0.60–0.70 for the best of
them is a good result and means the EGNN has real signal to combine. If the best
feature sits at 0.52, say so out loud rather than hoping the network rescues it;
the informative follow-up is then an ablation (Phase 5) comparing the model with
and without `x_topo`. The raw unstandardised features are kept on `d.x_topo`
precisely so that ablation is a one-line change.

### Next

**Phase 3 — `TopologicalEGNN`.** Equivariant convolution layers over
$(\mathbf{X}, \mathbf{H}, \mathcal{E})$, with the topological block fused by
cross-attention rather than plain concatenation, and a per-residue sigmoid head.